In [4]:
# Begin demo

In [11]:
%pip install xgboost

  Using cached scipy-1.15.3-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.7/131.7 MB 164.2 MB/s  0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 351.5/351.5 MB 105.3 MB/s  0:00:020:00:0100:01
Using cached scipy-1.15.3-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (37.7 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [xgboost]m2/3 [xgboost]ccl-cu12]
Note: you may need to restart the kernel to use updated packages.


In [10]:
%pip install numpy

Note: you may need to restart the kernel to use updated packages.


In [3]:
%pip install -U scikit-learn

  Using cached scikit_learn-1.7.2-cp310-cp310-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (11 kB)
  Using cached joblib-1.6.0-py3-none-any.whl.metadata (6.5 kB)
Using cached scikit_learn-1.7.2-cp310-cp310-manylinux2014_x86_64.manylinux_2_17_x86_64.whl (9.7 MB)
Using cached joblib-1.6.0-py3-none-any.whl (306 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [scikit-learn] [scikit-learn]
Note: you may need to restart the kernel to use updated packages.


In [19]:
import pandas as pd
import boto3
import io
import gzip
import json
from sklearn.model_selection import TimeSeriesSplit
import xgboost as xgb
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [13]:
# Create Dataframe

s3_client = boto3.client("s3")
bucket = "s3-3601-final-project"
prefix = "tomtom/traffic/"

response = s3_client.list_objects_v2(Bucket=bucket, Prefix=prefix)
df_list = []

if "Contents" in response:
    for obj in response["Contents"]:
        key = obj["Key"]

        if not key.endswith(".json.gz"):
            continue

        file_obj = s3_client.get_object(Bucket=bucket, Key=key)
        raw_bytes = file_obj["Body"].read()
        try:
            data_bytes = gzip.decompress(raw_bytes)
        except gzip.BadGzipFile:
            data_bytes = raw_bytes

        obj_df = pd.read_csv(io.BytesIO(data_bytes))
        df_list.append(obj_df)
        
        final_df = pd.concat(df_list, ignore_index=True)

    if df_list:
        final_df = pd.concat(df_list, ignore_index=True)
        final_df.head()
final_df.head(12)

,timestamp,latitude,longitude,current_speed,free_flow_speed,current_delay,confidence,day_of_week,hour,month,speed_ratio,holiday
0,2026-09-26 19:03:46,35.307778,-80.724583,19,27,NaN,1,5,19,9,0.703704,Not a federal holiday
1,2026-09-26 19:03:46,35.313611,-80.701944,40,40,NaN,1,5,19,9,1.000000,Not a federal holiday
2,2026-09-26 19:03:46,35.311361,-80.711222,26,31,NaN,1,5,19,9,0.838710,Not a federal holiday
3,2026-09-26 19:03:47,35.309306,-80.718056,26,31,NaN,1,5,19,9,0.838710,Not a federal holiday
4,2026-09-26 19:03:47,35.306917,-80.722639,26,31,NaN,1,5,19,9,0.838710,Not a federal holiday
5,2026-09-26 19:03:47,35.293111,-80.749361,17,28,NaN,1,5,19,9,0.607143,Not a federal holiday
6,2026-09-26 19:03:47,35.294889,-80.742111,55,55,NaN,1,5,19,9,1.000000,Not a federal holiday
7,2026-09-26 19:03:47,35.299639,-80.745750,19,28,NaN,1,5,19,9,0.678571,Not a federal holiday
8,2026-09-26 19:03:47,35.287972,-80.759472,20,25,NaN,1,5,19,9,0.800000,Not a federal holiday
9,2026-09-26 19:03:47,35.301861,-80.752333,18,26,NaN,1,5,19,9,0.692308,Not a federal holiday


In [ ]:
s3_client = boto3.client("s3")

response2 = s3_client.get_object(Bucket=bucket, Key="ncdot/traffic/aadt_stations_2026-09-27_15-13-28-302554.csv")


raw_bytes = response2["Body"].read()
try:
    data_bytes = gzip.decompress(raw_bytes)
except gzip.BadGzipFile:
    data_bytes = raw_bytes

df2_final = pd.read_csv(io.BytesIO(data_bytes))


df2_final.head(20)

In [14]:
# Edit the coordinates to match completely

# Rebuilt to match indices 0 through 10 of your df2_final perfectly
matched_latitudes = [
    35.287972,  # Row 0: EAST OF STETSON DR
    35.311361,  # Row 1: WEST OF I-485
    35.293111,  # Row 2: SOUTH OF UNIVERSITY POINTE BLVD
    35.309306,  # Row 3: EAST OF SUTHER RD
    35.306917,  # Row 4: WEST OF SUTHER RD
    35.313611,  # Row 5: EAST OF SR 5394 MALLARD CK
    35.311861,  # Row 6: WEST OF W 7TH ST 
    35.301861,  # Row 7: SOUTH OF US 29
    35.294889,  # Row 8: EAST OF SR 5316
    35.299639,  # Row 9: EAST OF SR 2939
    35.307778   # Row 10: NORTH OF NC 49
]

matched_longitudes = [
    -80.759472, # Row 0
    -80.711222, # Row 1
    -80.749361, # Row 2
    -80.718056, # Row 3
    -80.722639, # Row 4
    -80.701944, # Row 5
    -80.744000, # Row 6 
    -80.752333, # Row 7
    -80.742111, # Row 8
    -80.745750, # Row 9
    -80.724583  # Row 10
]

# Run your loop to update df2_final
for i in range(len(matched_latitudes)):
    df2_final.at[i, "matched_latitude"] = matched_latitudes[i]
    df2_final.at[i, "matched_longitude"] = matched_longitudes[i]


df2_final.head(20)

,LocationID,County,Route,Location,AADT_2022,longitude,latitude,matched_latitude,matched_longitude
0,600000044,MECKLENBURG,US 29/NC 49,EAST OF STETSON DR,29500,-80.759574,35.287985,35.287972,-80.759472
1,600000063,MECKLENBURG,NC 49,WEST OF I-485,32500,-80.711385,35.311292,35.311361,-80.711222
2,600000065,MECKLENBURG,US 29,SOUTH OF UNIVERSITY POINTE BLVD,18000,-80.755360,35.294440,35.293111,-80.749361
3,600000107,MECKLENBURG,NC 49,EAST OF SUTHER RD,26500,-80.727801,35.303270,35.309306,-80.718056
4,600000108,MECKLENBURG,NC 49,WEST OF SUTHER RD,27500,-80.729977,35.301876,35.306917,-80.722639
5,600000145,MECKLENBURG,NC 49,EAST OF SR 5394 MALLARD CK CH RD,42000,-80.717633,35.309346,35.313611,-80.701944
6,600000305,MECKLENBURG,US 29,WEST OF W 7TH ST,17000,-80.844289,35.232562,35.311861,-80.744000
7,600000412,MECKLENBURG,NC 24 (WT HARRIS BLVD),SOUTH OF US 29,47500,-80.748333,35.304176,35.301861,-80.752333
8,600000414,MECKLENBURG,NC 24 (E WT HARRIS BLVD),EAST OF SR 5316,51500,-80.739913,35.293191,35.294889,-80.742111
9,600000829,MECKLENBURG,NC 49,EAST OF SR 2939,32500,-80.722975,35.306539,35.299639,-80.745750


In [15]:
# Combine the two dataframes

final_df["latitude"] = pd.to_numeric(final_df["latitude"])
final_df["longitude"] = pd.to_numeric(final_df["longitude"])
df2_final["matched_latitude"] = pd.to_numeric(df2_final["matched_latitude"])
df2_final["matched_longitude"] = pd.to_numeric(df2_final["matched_longitude"])

df2_final["round_key"] = (
    df2_final["matched_latitude"].round(4).astype(str) + "_" + 
    df2_final["matched_longitude"].round(4).astype(str)
)

traffic_map=df2_final.set_index("round_key")["AADT_2022"]

final_df["round_key"] = (
    final_df["latitude"].round(4).astype(str) + "_" + 
    final_df["longitude"].round(4).astype(str)
)

final_df["AADT_2022"] = final_df["round_key"].map(traffic_map)

final_df = final_df.drop(columns=["round_key"])
df2_final = df2_final.drop(columns=["round_key"])

final_df.head(25)

,timestamp,latitude,longitude,current_speed,free_flow_speed,current_delay,confidence,day_of_week,hour,month,speed_ratio,holiday,AADT_2022
0,2026-09-26 19:03:46,35.307778,-80.724583,19,27,NaN,1,5,19,9,0.703704,Not a federal holiday,15500
1,2026-09-26 19:03:46,35.313611,-80.701944,40,40,NaN,1,5,19,9,1.000000,Not a federal holiday,42000
2,2026-09-26 19:03:46,35.311361,-80.711222,26,31,NaN,1,5,19,9,0.838710,Not a federal holiday,32500
3,2026-09-26 19:03:47,35.309306,-80.718056,26,31,NaN,1,5,19,9,0.838710,Not a federal holiday,26500
4,2026-09-26 19:03:47,35.306917,-80.722639,26,31,NaN,1,5,19,9,0.838710,Not a federal holiday,27500
5,2026-09-26 19:03:47,35.293111,-80.749361,17,28,NaN,1,5,19,9,0.607143,Not a federal holiday,18000
6,2026-09-26 19:03:47,35.294889,-80.742111,55,55,NaN,1,5,19,9,1.000000,Not a federal holiday,51500
7,2026-09-26 19:03:47,35.299639,-80.745750,19,28,NaN,1,5,19,9,0.678571,Not a federal holiday,32500
8,2026-09-26 19:03:47,35.287972,-80.759472,20,25,NaN,1,5,19,9,0.800000,Not a federal holiday,29500
9,2026-09-26 19:03:47,35.301861,-80.752333,18,26,NaN,1,5,19,9,0.692308,Not a federal holiday,47500


In [16]:
final_df["location_id"] = final_df.groupby(["latitude", "longitude"]).ngroup()

final_df = final_df.sort_values("timestamp").reset_index(drop=True)

final_df["is_holiday"] = (final_df["holiday"] != "Not a federal holiday").astype(int)

X = final_df[["location_id", "day_of_week", "hour", "month", "free_flow_speed", "is_holiday", "AADT_2022"]]
y = final_df["current_speed"]

split_idx = int(len(X) * 0.8)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

model = xgb.XGBRegressor(n_estimators=100, max_depth=5, learning_rate=0.1)
model.fit(X_train, y_train)

,objective,'reg:squarederror'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


In [28]:
y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

y_test_stable = np.where(y_test == 0, 1e-5, y_test)
mape = np.mean(np.abs((y_test - y_pred) / y_test_stable)) * 100

print("Model Performance:")
print(f"Mean Absolute Error (MAE): {mae:.2f} mph")
print(f"Root Mean Squared Error (RMSE): {rmse:.2f} mph")
print(f"MAPE (Percentage Error): {mape:.2f}%")
print(f"R² Score (Variance Explained): {r2:.4f}")

for i in range(len(y_pred)):
    print(f"Input:\n{X_test.iloc[i]} \nOutput speed: {y_pred[i]} mph")

Model Performance:
Mean Absolute Error (MAE): 2.36 mph
Root Mean Squared Error (RMSE): 4.08 mph
MAPE (Percentage Error): 10.49%
R² Score (Variance Explained): 0.7651
Input:
location_id            4
day_of_week            6
hour                   4
month                  9
free_flow_speed       27
is_holiday             1
AADT_2022          47500
Name: 52, dtype: int64 
Output speed: 27.014659881591797 mph
Input:
location_id            9
day_of_week            6
hour                   4
month                  9
free_flow_speed       29
is_holiday             1
AADT_2022          17000
Name: 53, dtype: int64 
Output speed: 30.268465042114258 mph
Input:
location_id            9
day_of_week            6
hour                   4
month                  9
free_flow_speed       29
is_holiday             0
AADT_2022          17000
Name: 54, dtype: int64 
Output speed: 30.268465042114258 mph
Input:
location_id            5
day_of_week            6
hour                  15
month                  